# 모듈 4. SQL/Python Tool 최소 권한

이 노트북은 LangChain SQL Tool과 `StructuredTool`을 사용해 SQL/Python 실행 도구에 최소 권한 원칙을 적용합니다.

## 학습 목표

- LangChain `QuerySQLDatabaseTool`이 강력한 만큼 실행 전 정책이 필요함을 이해한다.
- SQL Tool을 `SELECT` 전용으로 감싸는 안전 래퍼를 만든다.
- Python 실행 Tool에 금지 패턴, 제한된 builtins, timeout을 적용한다.
- 보안 없는 A안과 보안 적용 B안을 비교한다.

## 이번 모듈에서 사용하는 SQL

실습 DB는 **SQLite 파일 DB**입니다. `.env.example`의 `SECURITY_DEMO_SQLITE_PATH` 값을 읽어 DB 파일 경로를 정하고, LangChain의 `SQLDatabase`와 `QuerySQLDatabaseTool`로 연결합니다.

SQLite를 쓰는 이유는 별도 서버나 Docker 없이 SQL Tool 보안 구조를 보여주기 쉽기 때문입니다. 실제 운영에서는 PostgreSQL/MySQL 계정 권한, 읽기 전용 유저, Row/Column Level Security까지 함께 설계해야 합니다.

In [ ]:
import concurrent.futures
import os
import sqlite3
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import StructuredTool


load_dotenv()

DB_PATH = Path(os.getenv("SECURITY_DEMO_SQLITE_PATH", "security_demo.sqlite"))


def reset_demo_db() -> None:
    conn = sqlite3.connect(DB_PATH)
    conn.executescript(
        '''
        DROP TABLE IF EXISTS employees;
        CREATE TABLE employees (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            salary INTEGER NOT NULL
        );
        INSERT INTO employees (name, salary) VALUES
            ('Alice', 7000),
            ('Bob', 6500),
            ('Carol', 7200);
        '''
    )
    conn.close()


def make_sql_tool() -> QuerySQLDatabaseTool:
    db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH.as_posix()}")
    return QuerySQLDatabaseTool(db=db)

## A안. LangChain SQL Tool을 그대로 노출하기

`QuerySQLDatabaseTool` 자체는 강력한 실행 도구입니다. 실행 전에 애플리케이션 정책을 두지 않으면 모델이 생성한 `DROP TABLE` 같은 쿼리도 도구까지 도달할 수 있습니다.

In [ ]:
reset_demo_db()
raw_sql_tool = make_sql_tool()

print("정상 조회:")
print(raw_sql_tool.invoke({"query": "SELECT name, salary FROM employees"}))

print("\n위험 쿼리:")
print(raw_sql_tool.invoke({"query": "DROP TABLE employees"}))

print("\nDROP 이후 조회:")
print(raw_sql_tool.invoke({"query": "SELECT * FROM employees"}))

## B안. SELECT 전용 SQL Tool로 감싸기

안전 버전은 LangChain Tool 앞에 검증 함수를 둡니다. 이 예시는 쉬운 키워드 방식이지만, 실무에서는 SQL 파서와 DB 계정 권한을 함께 써야 합니다.

In [ ]:
FORBIDDEN_SQL_KEYWORDS = ["drop", "delete", "update", "insert", "alter", "truncate", "create"]


def validate_select_only(query: str) -> tuple[bool, str]:
    normalized = " ".join(query.lower().split())
    if not normalized.startswith("select "):
        return False, "SELECT 문만 허용됩니다."
    if ";" in normalized.rstrip(";"):
        return False, "여러 SQL 문장을 한 번에 실행할 수 없습니다."
    if any(keyword in normalized for keyword in FORBIDDEN_SQL_KEYWORDS):
        return False, "쓰기/삭제/DDL 키워드는 허용되지 않습니다."
    return True, "허용"


def safe_sql_query(query: str) -> str:
    allowed, reason = validate_select_only(query)
    if not allowed:
        return f"거부: {reason}"
    return raw_sql_tool.invoke({"query": query})


reset_demo_db()
raw_sql_tool = make_sql_tool()

safe_sql_tool = StructuredTool.from_function(
    func=safe_sql_query,
    name="safe_sql_query",
    description="SELECT 전용 SQL 조회 Tool. 쓰기/삭제/DDL 쿼리는 거부한다.",
)

print("위험 쿼리:", safe_sql_tool.invoke({"query": "DROP TABLE employees"}))
print("정상 쿼리:", safe_sql_tool.invoke({"query": "SELECT name, salary FROM employees ORDER BY salary DESC"}))

## Python Tool A/B 테스트

Python 실행 Tool도 SQL Tool만큼 위험합니다. 아래 예제는 `StructuredTool`로 실행 도구를 만들고, 안전 래퍼를 추가하는 방식입니다.

In [ ]:
SECRET_TOKEN = "demo-secret-token"


def unsafe_python_exec(code: str) -> str:
    namespace = {"SECRET_TOKEN": SECRET_TOKEN}
    exec(code, namespace)
    return str(namespace.get("answer", "answer 변수가 없습니다."))


unsafe_python_tool = StructuredTool.from_function(
    func=unsafe_python_exec,
    name="unsafe_python_exec",
    description="전달받은 Python 코드를 그대로 실행한다. 강의용 취약 예시.",
)

print(unsafe_python_tool.invoke({"code": "answer = SECRET_TOKEN"}))

In [ ]:
FORBIDDEN_PYTHON_PATTERNS = [
    "import ",
    "__",
    "open(",
    "os.",
    "subprocess",
    "socket",
    "eval(",
    "exec(",
    "SECRET_TOKEN",
]


def run_with_timeout(code: str, timeout_seconds: float = 1.0) -> str:
    def _run() -> str:
        namespace = {"answer": None}
        exec(code, {"__builtins__": {"len": len, "sum": sum, "range": range}}, namespace)
        return str(namespace.get("answer"))

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_run)
        return future.result(timeout=timeout_seconds)


def safe_python_exec(code: str) -> str:
    normalized = code.replace(" ", "")
    for pattern in FORBIDDEN_PYTHON_PATTERNS:
        if pattern.replace(" ", "") in normalized:
            return f"거부: 금지 패턴 감지 - {pattern}"
    try:
        return run_with_timeout(code)
    except concurrent.futures.TimeoutError:
        return "거부: 실행 시간이 너무 깁니다."
    except Exception as exc:
        return f"실행 오류: {exc}"


safe_python_tool = StructuredTool.from_function(
    func=safe_python_exec,
    name="safe_python_exec",
    description="제한된 Python 코드만 실행한다. 파일/네트워크/비밀값 접근은 거부한다.",
)

print("악성 코드:", safe_python_tool.invoke({"code": "answer = SECRET_TOKEN"}))
print("정상 코드:", safe_python_tool.invoke({"code": "answer = sum(range(10))"}))

## 정리

- LangChain Tool은 편리하지만 그 자체가 보안 정책은 아닙니다.
- SQL Tool은 최소한 SELECT 전용 래퍼와 읽기 전용 DB 계정이 필요합니다.
- Python Tool은 가능하면 운영 Agent에 직접 노출하지 말고, 필요하다면 격리 환경과 timeout, 파일/네트워크 제한을 함께 둬야 합니다.

## 실습 과제

1. `SELECT *`를 금지하고 필요한 컬럼만 허용하도록 `validate_select_only`를 개선해보세요.
2. `safe_python_exec`의 실행 로그에 코드 길이, 차단 사유, 실행 시간을 남겨보세요.